In [1]:
# =========================
# [Cell 1] 설정/라이브러리
# =========================
from pathlib import Path
from datetime import datetime
import os, json
import numpy as np
import pandas as pd

import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import sklearn
print("sklearn version:", sklearn.__version__)

# -------------------------
# 경로 설정
# -------------------------
DATA_DIR = Path(r"C:\Users\eys63\GitHub\FlavorGraph\맥주데이터분석\데이터분석\data")
XLSX_PATH = DATA_DIR / "Supplemental Files and Figure source files.xlsx"

# 결과 저장 폴더(현재시간 기반)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path(r"C:\Users\eys63\GitHub\FlavorGraph\맥주데이터분석\데이터분석\output\gpt_ver1") / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

# CPU 병렬 사용 (GridSearchCV에서 사용)
N_JOBS = os.cpu_count() or 4

# 논문/Zenodo 흐름에 맞춘 기본 seed
SPLIT_SEED = 0  # Zenodo 노트북의 기본 분할 random_state=0 흐름
RANDOM_STATE_MODEL = 0

print("OUT_DIR:", OUT_DIR)
print("N_JOBS:", N_JOBS)


sklearn version: 1.7.2
OUT_DIR: C:\Users\eys63\GitHub\FlavorGraph\맥주데이터분석\데이터분석\output\gpt_ver1\20260125_071136
N_JOBS: 16


In [2]:
# ============================================
# [Cell 2] 데이터 로드 (S1: 화학, S4: 관능)
# ============================================
# 논문/Zenodo에서 사용한 핵심 시트:
# - Supplementary File S1: chemical features (231개)
# - Supplementary File S4: trained panel sensory scores (50개)

s1 = pd.read_excel(XLSX_PATH, sheet_name="Supplementary File S1")
s4 = pd.read_excel(XLSX_PATH, sheet_name="Supplementary File S4")

# 공통 키(beer, beer_id, tasting_category_fine)로 머지
df = s1.merge(s4, on=["beer", "beer_id", "tasting_category_fine"], how="inner")

meta_cols = ["beer", "beer_id", "tasting_category_fine"]
chem_cols = list(s1.columns[3:])   # 231개 화학 변수
sensory_cols = list(s4.columns[3:]) # 50개 관능 변수

X = df[chem_cols].copy()
Y = df[sensory_cols].copy()
style = df["tasting_category_fine"].copy()  # beer style(맥주 스타일 라벨)

print("N samples:", len(df))
print("X shape:", X.shape, "Y shape:", Y.shape)
print("n_styles:", style.nunique())
print("min count per style:", style.value_counts().min())


N samples: 250
X shape: (250, 231) Y shape: (250, 50)
n_styles: 22
min count per style: 3


In [4]:
# ============================================================
# [Cell 3] (수정본) 논문 최고 성능 모델 직접 학습:
#         - 관능(descriptor: 관능 항목) 50개 각각에 대해
#           GradientBoostingRegressor(GBR: 그래디언트 부스팅 회귀) GridSearchCV 실행
#         - Train/Test R^2, RMSE 기록
# ============================================================

import os, json, inspect
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

import joblib

# ----------------------------
# (0) 필수 변수 존재 체크
# ----------------------------
required_vars = ["X", "Y", "style", "sensory_cols", "OUT_DIR", "N_JOBS"]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(
        f"Cell 1~2를 먼저 실행해야 합니다. 누락 변수: {missing}\n"
        "필요 변수: X, Y, style, sensory_cols, OUT_DIR, N_JOBS"
    )

# ----------------------------
# (1) RMSE 계산 함수(버전 호환)
# ----------------------------
def rmse_score(y_true, y_pred):
    """
    RMSE(Root Mean Squared Error: 평균제곱오차의 제곱근)
    - scikit-learn 버전에 따라 mean_squared_error(..., squared=False)가 없을 수 있어
    - 그래서 항상 MSE를 구한 뒤 sqrt로 RMSE를 만든다(버전 무관).
    """
    mse = mean_squared_error(y_true, y_pred)  # MSE(Mean Squared Error: 평균제곱오차)
    return float(np.sqrt(mse))

def eval_regression(y_true, y_pred):
    """회귀(regression: 연속값 예측) 성능지표: R^2, RMSE"""
    r2 = float(r2_score(y_true, y_pred))
    rmse = rmse_score(y_true, y_pred)
    return r2, rmse

def to_builtin(obj):
    """numpy 타입을 json 저장 가능한 파이썬 기본 타입으로 변환"""
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    return obj

# ----------------------------
# (2) Train/Test split (층화 분할, stratified split: 스타일 비율 유지)
# ----------------------------
SPLIT_SEED = 0  # 논문/Zenodo 흐름과 맞추는 random_state(랜덤 시드)
idx_all = np.arange(len(X))

idx_train, idx_test = train_test_split(
    idx_all,
    test_size=0.30,
    random_state=SPLIT_SEED,
    shuffle=True,
    stratify=style
)

X_train = X.iloc[idx_train].copy()
X_test  = X.iloc[idx_test].copy()
Y_train = Y.iloc[idx_train].copy()
Y_test  = Y.iloc[idx_test].copy()

style_train = style.iloc[idx_train].copy()
style_test  = style.iloc[idx_test].copy()

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("n_styles(train/test):", style_train.nunique(), style_test.nunique())

# ----------------------------
# (3) scikit-learn 버전에 따라 GBR loss 이름이 다를 수 있어 자동 대응
# ----------------------------
def get_gbr_loss_names():
    """
    일부 scikit-learn 버전에서는
    - 'squared_error', 'absolute_error'
    대신
    - 'ls', 'lad'
    를 쓴다.
    """
    try:
        _ = GradientBoostingRegressor(loss="squared_error")
        return ["squared_error", "absolute_error"]
    except Exception:
        return ["ls", "lad"]

LOSS_LIST = get_gbr_loss_names()
print("GBR loss candidates:", LOSS_LIST)

# ----------------------------
# (4) 논문/Zenodo 기준 GradientBoostingRegressor 탐색 공간(grid)
# ----------------------------
param_grid_gbr = [{
    "model__subsample": np.arange(0.2, 1.0, 0.2),       # 0.2,0.4,0.6,0.8
    "model__learning_rate": np.arange(0.02, 0.2, 0.02), # 0.02~0.18
    "model__max_depth": np.arange(6, 15, 3),            # 6,9,12
    "model__n_estimators": [200, 500, 1000],
    "model__loss": LOSS_LIST,
}]

# ----------------------------
# (5) 파이프라인(pipeline: 전처리+모델 묶음)
# ----------------------------
base_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),  # mean imputation(평균 대치)
    ("scaler", StandardScaler()),                 # standardization(표준화)
    ("model", GradientBoostingRegressor(random_state=0)),
])

# ----------------------------
# (6) 실행 설정
# ----------------------------
# 전체 50개는 오래 걸릴 수 있어.
# 처음엔 10개만 테스트해도 됨.
DESCRIPTORS_TO_RUN = sensory_cols         # 전체 실행
# DESCRIPTORS_TO_RUN = sensory_cols[:10]  # 빠른 테스트용

VERBOSE = 0
CV_FOLDS = 5
SAVE_EACH = True

models_dir = Path(OUT_DIR) / "models_per_descriptor"
cv_dir     = Path(OUT_DIR) / "cv_results_per_descriptor"
models_dir.mkdir(parents=True, exist_ok=True)
cv_dir.mkdir(parents=True, exist_ok=True)

results = []

for i, col in enumerate(DESCRIPTORS_TO_RUN, 1):
    print(f"\n[{i}/{len(DESCRIPTORS_TO_RUN)}] Training descriptor = {col}")

    ytr = Y_train[col].values
    yte = Y_test[col].values

    gs = GridSearchCV(
        estimator=base_pipe,
        param_grid=param_grid_gbr,
        scoring="r2",
        cv=CV_FOLDS,
        n_jobs=N_JOBS,
        verbose=VERBOSE,
        refit=True,
        return_train_score=True,
        error_score="raise"  # 조용히 NaN으로 넘기지 않고 바로 오류로 보여줌(디버깅에 유리)
    )

    gs.fit(X_train, ytr)

    best_pipe = gs.best_estimator_
    pred_tr = best_pipe.predict(X_train)
    pred_te = best_pipe.predict(X_test)

    r2_tr, rmse_tr = eval_regression(ytr, pred_tr)
    r2_te, rmse_te = eval_regression(yte, pred_te)

    best_params = {k: to_builtin(v) for k, v in gs.best_params_.items()}

    row = {
        "descriptor": col,
        "r2_train": r2_tr,
        "rmse_train": rmse_tr,
        "r2_test": r2_te,
        "rmse_test": rmse_te,
        "best_params_json": json.dumps(best_params, ensure_ascii=False),
    }
    results.append(row)

    if SAVE_EACH:
        joblib.dump(best_pipe, models_dir / f"GBR_best__{col}.joblib")
        pd.DataFrame(gs.cv_results_).to_csv(cv_dir / f"GBR_cv__{col}.csv", index=False)

    print(f"  -> BEST test R2={r2_te:.4f}, test RMSE={rmse_te:.4f}")
    print(f"  -> best_params={best_params}")

df_perf = pd.DataFrame(results).sort_values("r2_test", ascending=False)
df_perf.to_csv(Path(OUT_DIR) / "GBR__per_descriptor_train_test_scores.csv", index=False)

print("\n===== Top 10 descriptors by TEST R2 =====")
display(df_perf.head(10))

config = {
    "SPLIT_SEED": SPLIT_SEED,
    "CV_FOLDS": CV_FOLDS,
    "DESCRIPTORS_RAN": list(DESCRIPTORS_TO_RUN),
    "loss_candidates": LOSS_LIST,
    "param_grid_gbr": {
        "subsample": list(np.arange(0.2, 1.0, 0.2)),
        "learning_rate": list(np.arange(0.02, 0.2, 0.02)),
        "max_depth": list(np.arange(6, 15, 3)),
        "n_estimators": [200, 500, 1000],
        "loss": LOSS_LIST
    }
}
with open(Path(OUT_DIR) / "GBR_descriptor_training_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)


Train size: (175, 231) Test size: (75, 231)
n_styles(train/test): 22 22
GBR loss candidates: ['squared_error', 'absolute_error']

[1/50] Training descriptor = A_malt_all
  -> BEST test R2=0.3293, test RMSE=0.6608
  -> best_params={'model__learning_rate': 0.08, 'model__loss': 'absolute_error', 'model__max_depth': 9, 'model__n_estimators': 500, 'model__subsample': 0.8}

[2/50] Training descriptor = A_malt_grain
  -> BEST test R2=0.0210, test RMSE=0.4191
  -> best_params={'model__learning_rate': 0.02, 'model__loss': 'absolute_error', 'model__max_depth': 9, 'model__n_estimators': 500, 'model__subsample': 0.8}

[3/50] Training descriptor = A_malt_bread
  -> BEST test R2=-0.0106, test RMSE=0.4784
  -> best_params={'model__learning_rate': 0.02, 'model__loss': 'absolute_error', 'model__max_depth': 6, 'model__n_estimators': 200, 'model__subsample': 0.6000000000000001}

[4/50] Training descriptor = A_malt_cara
  -> BEST test R2=0.3393, test RMSE=0.4893
  -> best_params={'model__learning_rate': 0

KeyboardInterrupt: 

In [ ]:
# ============================================================
# [Cell 4] 가장 잘 예측되는 관능 1개를 TARGET_Y로 선택
# ============================================================

import json
from pathlib import Path
import joblib
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

def rmse_score(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return float(np.sqrt(mse))

# df_perf가 없다면 저장된 csv에서 로드
if "df_perf" not in globals():
    score_path = Path(OUT_DIR) / "GBR__per_descriptor_train_test_scores.csv"
    df_perf = pd.read_csv(score_path)

best_row = df_perf.sort_values("r2_test", ascending=False).iloc[0]
TARGET_Y = best_row["descriptor"]

print("Selected TARGET_Y:", TARGET_Y)
print("Best test R2:", best_row["r2_test"], "Best test RMSE:", best_row["rmse_test"])

best_params = json.loads(best_row["best_params_json"])
print("Best params:", best_params)

best_model_path = Path(OUT_DIR) / "models_per_descriptor" / f"GBR_best__{TARGET_Y}.joblib"
best_model_loaded = joblib.load(best_model_path)

yte = Y_test[TARGET_Y].values
pred_te = best_model_loaded.predict(X_test)
r2_te = float(r2_score(yte, pred_te))
rmse_te = rmse_score(yte, pred_te)

print(f"[Sanity check(간단 재검증)] Loaded model test R2={r2_te:.4f}, RMSE={rmse_te:.4f}")

with open(Path(OUT_DIR) / "TARGET_Y.json", "w", encoding="utf-8") as f:
    json.dump({"TARGET_Y": TARGET_Y, "best_params": best_params}, f, ensure_ascii=False, indent=2)


In [ ]:
# ============================================================
# [Cell 5] 스타일(style: 맥주 스타일) 제거 vs 동일 수 랜덤 제거 비교
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

def rmse_score(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return float(np.sqrt(mse))

def make_best_gbr_pipeline(best_params_dict, random_state=0):
    model_params = {}
    for k, v in best_params_dict.items():
        if k.startswith("model__"):
            model_params[k.replace("model__", "")] = v

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("model", GradientBoostingRegressor(random_state=random_state, **model_params)),
    ])
    return pipe

def compute_metrics(pipe, X_tr, y_tr, X_te, y_te):
    pipe.fit(X_tr, y_tr)
    pred_tr = pipe.predict(X_tr)
    pred_te = pipe.predict(X_te)
    return {
        "r2_train": float(r2_score(y_tr, pred_tr)),
        "rmse_train": rmse_score(y_tr, pred_tr),
        "r2_test": float(r2_score(y_te, pred_te)),
        "rmse_test": rmse_score(y_te, pred_te),
    }

y_train = Y_train[TARGET_Y]
y_test  = Y_test[TARGET_Y]

base_pipe = make_best_gbr_pipeline(best_params, random_state=0)
base_metrics = compute_metrics(base_pipe, X_train, y_train, X_test, y_test)
print("BASE:", base_metrics)

N_RANDOM = 100  # random repeats(랜덤 반복)
rng = np.random.RandomState(0)

style_results = []
styles = style_train.unique().tolist()

for st in styles:
    idx_remove = np.where(style_train.values == st)[0]
    n_remove = len(idx_remove)
    if n_remove == 0:
        continue

    # 그룹 제거(group ablation: 특정 그룹을 통째로 제외)
    keep_mask = np.ones(len(X_train), dtype=bool)
    keep_mask[idx_remove] = False

    pipe_st = make_best_gbr_pipeline(best_params, random_state=0)
    met_st = compute_metrics(pipe_st, X_train.iloc[keep_mask], y_train.iloc[keep_mask], X_test, y_test)

    delta_r2 = base_metrics["r2_test"] - met_st["r2_test"]
    delta_rmse = met_st["rmse_test"] - base_metrics["rmse_test"]

    # 동일 수 랜덤 제거(random removal: 무작위로 같은 개수 제거) 분포
    deltas_r2 = []
    deltas_rmse = []
    for _ in range(N_RANDOM):
        ridx = rng.choice(len(X_train), size=n_remove, replace=False)
        keep = np.ones(len(X_train), dtype=bool)
        keep[ridx] = False

        pipe_r = make_best_gbr_pipeline(best_params, random_state=0)
        met_r = compute_metrics(pipe_r, X_train.iloc[keep], y_train.iloc[keep], X_test, y_test)

        deltas_r2.append(base_metrics["r2_test"] - met_r["r2_test"])
        deltas_rmse.append(met_r["rmse_test"] - base_metrics["rmse_test"])

    deltas_r2 = np.array(deltas_r2)
    deltas_rmse = np.array(deltas_rmse)

    pct_r2 = float(np.mean(deltas_r2 <= delta_r2) * 100.0)
    pct_rmse = float(np.mean(deltas_rmse <= delta_rmse) * 100.0)

    style_results.append({
        "group_type": "style",
        "group_name": st,
        "n_remove": int(n_remove),

        "base_r2_test": base_metrics["r2_test"],
        "after_r2_test": met_st["r2_test"],
        "delta_r2_drop": float(delta_r2),

        "base_rmse_test": base_metrics["rmse_test"],
        "after_rmse_test": met_st["rmse_test"],
        "delta_rmse_increase": float(delta_rmse),

        "random_r2_drop_mean": float(deltas_r2.mean()),
        "random_r2_drop_std": float(deltas_r2.std()),
        "random_rmse_inc_mean": float(deltas_rmse.mean()),
        "random_rmse_inc_std": float(deltas_rmse.std()),

        "percentile_r2_drop_vs_random": pct_r2,
        "percentile_rmse_inc_vs_random": pct_rmse,
    })

style_df = pd.DataFrame(style_results).sort_values("delta_r2_drop", ascending=False)
style_df.to_csv(Path(OUT_DIR) / f"ablation_style__{TARGET_Y}.csv", index=False)

display(style_df.head(10))


In [ ]:
# ============================================================
# [Cell 6] 클러스터(cluster: 자동으로 묶인 그룹) 제거 vs 랜덤 제거
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

y_train = Y_train[TARGET_Y]
y_test  = Y_test[TARGET_Y]

# baseline
base_pipe = make_best_gbr_pipeline(best_params, random_state=0)
base_metrics = compute_metrics(base_pipe, X_train, y_train, X_test, y_test)
print("BASE:", base_metrics)

# clustering용 전처리(preprocessing: 결측 대치 + 표준화)
prep = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])
X_train_prep = prep.fit_transform(X_train)

# PCA(차원축소): 고차원 거리 불안정 문제 완화
PCA_DIM = 20
pca = PCA(n_components=min(PCA_DIM, X_train_prep.shape[1]), random_state=0)
X_train_pca = pca.fit_transform(X_train_prep)

# K 선택: silhouette score(군집 품질 지표) 최대
K_RANGE = range(2, 11)
sil_scores = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=0, n_init=20)
    labels = km.fit_predict(X_train_pca)
    sil_scores.append((k, float(silhouette_score(X_train_pca, labels))))

best_k, best_sil = sorted(sil_scores, key=lambda x: x[1], reverse=True)[0]
print("best_k:", best_k, "best_silhouette:", best_sil)
print("all silhouette:", sil_scores)

kmeans = KMeans(n_clusters=best_k, random_state=0, n_init=20)
cluster_labels = kmeans.fit_predict(X_train_pca)

N_RANDOM = 100
rng = np.random.RandomState(0)

cluster_results = []
for c in range(best_k):
    idx_remove = np.where(cluster_labels == c)[0]
    n_remove = len(idx_remove)
    if n_remove == 0:
        continue

    # 클러스터 제거
    keep_mask = np.ones(len(X_train), dtype=bool)
    keep_mask[idx_remove] = False

    pipe_c = make_best_gbr_pipeline(best_params, random_state=0)
    met_c = compute_metrics(pipe_c, X_train.iloc[keep_mask], y_train.iloc[keep_mask], X_test, y_test)

    delta_r2 = base_metrics["r2_test"] - met_c["r2_test"]
    delta_rmse = met_c["rmse_test"] - base_metrics["rmse_test"]

    # 동일 수 랜덤 제거 분포
    deltas_r2 = []
    deltas_rmse = []
    for _ in range(N_RANDOM):
        ridx = rng.choice(len(X_train), size=n_remove, replace=False)
        keep = np.ones(len(X_train), dtype=bool)
        keep[ridx] = False

        pipe_r = make_best_gbr_pipeline(best_params, random_state=0)
        met_r = compute_metrics(pipe_r, X_train.iloc[keep], y_train.iloc[keep], X_test, y_test)

        deltas_r2.append(base_metrics["r2_test"] - met_r["r2_test"])
        deltas_rmse.append(met_r["rmse_test"] - base_metrics["rmse_test"])

    deltas_r2 = np.array(deltas_r2)
    deltas_rmse = np.array(deltas_rmse)

    pct_r2 = float(np.mean(deltas_r2 <= delta_r2) * 100.0)
    pct_rmse = float(np.mean(deltas_rmse <= delta_rmse) * 100.0)

    cluster_results.append({
        "group_type": "cluster",
        "cluster_id": int(c),
        "n_remove": int(n_remove),

        "base_r2_test": base_metrics["r2_test"],
        "after_r2_test": met_c["r2_test"],
        "delta_r2_drop": float(delta_r2),

        "base_rmse_test": base_metrics["rmse_test"],
        "after_rmse_test": met_c["rmse_test"],
        "delta_rmse_increase": float(delta_rmse),

        "random_r2_drop_mean": float(np.mean(deltas_r2)),
        "random_r2_drop_std": float(np.std(deltas_r2)),
        "random_rmse_inc_mean": float(np.mean(deltas_rmse)),
        "random_rmse_inc_std": float(np.std(deltas_rmse)),

        "percentile_r2_drop_vs_random": pct_r2,
        "percentile_rmse_inc_vs_random": pct_rmse,
    })

cluster_df = pd.DataFrame(cluster_results).sort_values("delta_r2_drop", ascending=False)
cluster_df.to_csv(Path(OUT_DIR) / f"ablation_cluster__{TARGET_Y}.csv", index=False)

display(cluster_df.head(10))
